## Preprocessing OpenNQ dataset into an evaluation split

In [18]:
import json, os, random

In [19]:
data =[]
with open('../data/raw/OpenNQ/nq-dev-all.jsonl', 'r') as file:
    for line in file:
        data.append(json.loads(line))
    

In [20]:
random.seed(42)

In [21]:
len(data)

7830

In [22]:
for i, datapoint in enumerate(data):
    datapoint['datapoint_id'] = i

In [23]:
data[0].keys()

dict_keys(['annotations', 'document_html', 'document_title', 'document_tokens', 'document_url', 'example_id', 'long_answer_candidates', 'question_text', 'question_tokens', 'datapoint_id'])

In [24]:
data[0]['annotations'][0]

{'annotation_id': 13591449469826568799,
 'long_answer': {'candidate_index': 92,
  'end_byte': 67824,
  'end_token': 925,
  'start_byte': 66429,
  'start_token': 808},
 'short_answers': [{'end_byte': 66817,
   'end_token': 837,
   'start_byte': 66588,
   'start_token': 816}],
 'yes_no_answer': 'NONE'}

In [25]:
data_with_single_short_answer = []
for datapoint in data:
    for i, annotation in enumerate(datapoint['annotations']):
        if len(annotation['short_answers']) == 1:
            datapoint['picked_annotation'] = {'annotation_number': i, 'type': 'short_answer', 'answer': annotation['short_answers'][0]}
            data_with_single_short_answer.append(datapoint)
            break
        elif annotation['yes_no_answer'] != 'NONE':
            datapoint['picked_annotation'] = {'annotation_number': i, 'type': 'yes_no', 'answer': annotation['yes_no_answer'].lower()}
            data_with_single_short_answer.append(datapoint)
            break


In [26]:
len(data_with_single_short_answer)

4114

In [27]:
data_sample = random.sample(data_with_single_short_answer, 2000)

In [28]:
def extract_short_answer(start_token, end_token, document_tokens):
    tokens =[]
    for i in range(start_token, end_token):
        tokens.append(document_tokens[i]['token'])
    return ' '.join(tokens)

In [29]:
sample_points = []
for i, datapoint in enumerate(data_sample):
    sample_point ={"question_id": i, "datapoint_id": datapoint['datapoint_id'], "question": datapoint['question_text'], "annotation_id": datapoint['picked_annotation']['annotation_number'], "picked_annotation_type": datapoint['picked_annotation']['type']}
    if datapoint['picked_annotation']['type'] == 'short_answer':
        sample_point['gt_answer'] = extract_short_answer(datapoint['picked_annotation']['answer']['start_token'], datapoint['picked_annotation']['answer']['end_token'], datapoint['document_tokens'])
    else:
        sample_point['gt_answer'] = datapoint['picked_annotation']['answer']
    sample_points.append(sample_point)

In [30]:
sample_points

[{'question_id': 0,
  'datapoint_id': 1716,
  'question': 'when does the little couples new season start',
  'annotation_id': 0,
  'picked_annotation_type': 'short_answer',
  'gt_answer': 'September 19 , 2017'},
 {'question_id': 1,
  'datapoint_id': 387,
  'question': 'who wins the next iron chef super chefs',
  'annotation_id': 1,
  'picked_annotation_type': 'short_answer',
  'gt_answer': 'Geoffrey Zakarian'},
 {'question_id': 2,
  'datapoint_id': 4271,
  'question': 'who has the best nba record this season',
  'annotation_id': 2,
  'picked_annotation_type': 'short_answer',
  'gt_answer': 'Houston Rockets'},
 {'question_id': 3,
  'datapoint_id': 3811,
  'question': 'north carolina delegate to the second continental congress',
  'annotation_id': 2,
  'picked_annotation_type': 'short_answer',
  'gt_answer': 'John B. Ashe'},
 {'question_id': 4,
  'datapoint_id': 3478,
  'question': 'when is the 5th round fa cup played',
  'annotation_id': 0,
  'picked_annotation_type': 'short_answer',
  

In [31]:
final_dataset = []
for sample_point in sample_points:
    final_datapoint = {
        "question_id": sample_point['question_id'],
        "datapoint_id": sample_point['datapoint_id'],
        "question": sample_point['question'],
        "gt_answer": sample_point['gt_answer'],
        "annotation_idx": sample_point['annotation_id'],
        "annotation_type": sample_point['picked_annotation_type']
    }
    
    final_dataset.append(final_datapoint)

In [32]:
final_dataset

[{'question_id': 0,
  'datapoint_id': 1716,
  'question': 'when does the little couples new season start',
  'gt_answer': 'September 19 , 2017',
  'annotation_idx': 0,
  'annotation_type': 'short_answer'},
 {'question_id': 1,
  'datapoint_id': 387,
  'question': 'who wins the next iron chef super chefs',
  'gt_answer': 'Geoffrey Zakarian',
  'annotation_idx': 1,
  'annotation_type': 'short_answer'},
 {'question_id': 2,
  'datapoint_id': 4271,
  'question': 'who has the best nba record this season',
  'gt_answer': 'Houston Rockets',
  'annotation_idx': 2,
  'annotation_type': 'short_answer'},
 {'question_id': 3,
  'datapoint_id': 3811,
  'question': 'north carolina delegate to the second continental congress',
  'gt_answer': 'John B. Ashe',
  'annotation_idx': 2,
  'annotation_type': 'short_answer'},
 {'question_id': 4,
  'datapoint_id': 3478,
  'question': 'when is the 5th round fa cup played',
  'gt_answer': 'February 2018',
  'annotation_idx': 0,
  'annotation_type': 'short_answer'},

In [33]:
with open('../data/eval/OpenNQ_2k.json', 'w') as file:
    json.dump(final_dataset, file, indent=4) 

In [34]:

gt_answers = []
for entry in final_dataset:
    gt_answers.append({
        "question_id": entry["question_id"],
        "variant_id": 0,
        "answer_id": 0,
        "generated_answer": entry["gt_answer"]
    })
with open('../data/logs/OpenNQ_2k/gt-answers.json', 'w') as f:
    json.dump(gt_answers, f, indent=4)